In [6]:
import numpy as np
import nltk
import pickle
from nltk.corpus import brown
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Bidirectional, Dropout
from sklearn.model_selection import train_test_split

In [7]:
nltk.download('brown')

sentences = brown.sents()
sentences = [[word.lower() for word in sent] for sent in sentences if len(sent) > 5]

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [8]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=200,
    window=5,
    min_count=2,
    workers=4
)

In [9]:
mask_token = "mask"
mask_index = None

inputs = []
targets = []

for sentence in sentences:
    for i in range(1, len(sentence)-1):
        masked = sentence.copy()
        target_word = masked[i]
        masked[i] = mask_token

        inputs.append(" ".join(masked))
        targets.append(target_word)

In [10]:
tokenizer = Tokenizer(num_words=8000, oov_token="<OOV>")
tokenizer.fit_on_texts(inputs + targets)
mask_index = tokenizer.word_index.get(mask_token)

total_words = 8000

In [11]:
input_sequences = tokenizer.texts_to_sequences(inputs)
target_sequences = tokenizer.texts_to_sequences(targets)

In [12]:
filtered_inputs = []
filtered_targets = []

for inp, tar in zip(input_sequences, target_sequences):
    if len(tar) > 0 and tar[0] < total_words:
        filtered_inputs.append(inp)
        filtered_targets.append(tar[0])

In [13]:
max_sequence_len = max(len(seq) for seq in filtered_inputs)

X = pad_sequences(filtered_inputs, maxlen=max_sequence_len, padding='pre')
y = np.array(filtered_targets)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
embedding_dim = 200
embedding_matrix = np.zeros((total_words, embedding_dim))

for word, index in tokenizer.word_index.items():
    if index < total_words:
        if word in w2v_model.wv:
            embedding_matrix[index] = w2v_model.wv[word]

In [16]:
model = Sequential()

model.add(Embedding(
    input_dim=total_words,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=max_sequence_len,
    mask_zero=True,         
    trainable=True
))

model.add(Bidirectional(GRU(256,return_sequences=True,recurrent_dropout=0.2)))
model.add(Dropout(0.3))

model.add(Bidirectional(GRU(256,recurrent_dropout=0.2)))
model.add(Dropout(0.3))

model.add(Dense(total_words, activation='softmax'))

from tensorflow.keras.optimizers import Adam
optimizer = Adam(learning_rate=0.0005)

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

In [12]:
X_train_small = X_train[:200000]
y_train_small = y_train[:200000]

from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1,
    min_lr=1e-6
)

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

model.fit(
    X_train_small,
    y_train_small,
    epochs=10,
    batch_size=256,
    validation_data=(X_test[:50000], y_test[:50000]), 
    callbacks=[lr_scheduler, early_stop]
)

Epoch 1/10


782/782 [==============================] - 9705s 12s/step - loss: 6.2984 - accuracy: 0.1177 - val_loss: 5.6297 - val_accuracy: 0.1868 - lr: 5.0000e-04
Epoch 2/10
782/782 [==============================] - 9448s 12s/step - loss: 5.2865 - accuracy: 0.2110 - val_loss: 5.1418 - val_accuracy: 0.2240 - lr: 5.0000e-04
Epoch 3/10
782/782 [==============================] - 8634s 11s/step - loss: 4.8312 - accuracy: 0.2400 - val_loss: 4.9640 - val_accuracy: 0.2419 - lr: 5.0000e-04
Epoch 4/10
782/782 [==============================] - 8498s 11s/step - loss: 4.5210 - accuracy: 0.2592 - val_loss: 4.8806 - val_accuracy: 0.2496 - lr: 5.0000e-04
Epoch 5/10
782/782 [==============================] - 8564s 11s/step - loss: 4.2652 - accuracy: 0.2727 - val_loss: 4.8454 - val_accuracy: 0.2527 - lr: 5.0000e-04
Epoch 6/10
782/782 [==============================] - 8693s 11s/step - loss: 4.0404 - accuracy: 0.2854 - val_loss: 4.8459 - val_accuracy: 0.2532 - lr: 5.0000e-04
Epoch 7/10
782/782 [======

In [13]:
model.save("bigru_mask_model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved successfully!")

c:\Users\dell\anaconda3\envs\lstmvenv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model and tokenizer saved successfully!


In [17]:
def predict_masked_word(model, tokenizer, sentence, max_sequence_len, temperature=1.0):

    seq = tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=max_sequence_len, padding='pre')

    probs = model.predict(padded, verbose=0)[0]

    top_k = 15
    top_indices = np.argsort(probs)[-top_k:]
    filtered_probs = np.zeros_like(probs)
    filtered_probs[top_indices] = probs[top_indices]
    probs = filtered_probs / np.sum(filtered_probs)

    probs = np.log(probs + 1e-10) / temperature
    probs = np.exp(probs)
    probs = probs / np.sum(probs)

    prediction = np.random.choice(len(probs), p=probs)

    return tokenizer.index_word.get(prediction, "Word not found")

In [18]:
def predict_top_n(model, tokenizer, sentence, max_sequence_len, n=5):

    seq = tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=max_sequence_len, padding='pre')

    probs = model.predict(padded, verbose=0)[0]

    top_indices = np.argsort(probs)[-n:][::-1]

    return [(tokenizer.index_word.get(i, ""), probs[i]) for i in top_indices]

In [21]:
print(predict_masked_word(
    model,
    tokenizer,
    "my name is Ali, my hobby is traveling, Currently i am learning a course on Gen-AI and then i will mask for the job as AI-ML Engineer",
    max_sequence_len
))


exports


In [22]:
model = load_model("bigru_mask_model.h5")

NameError: name 'load_model' is not defined

In [1]:
dir bigru_mask_model.h5

SyntaxError: invalid syntax (1765850722.py, line 1)